# 32. 全履歴による特徴量の一致確認
出典：FX (3).ipynb、元セルindex [70, 71, 72, 73]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 70
構文状態：valid


In [ ]:
# ============================================================
# HISTORICAL REPLAY PARITY TEST v1
#
# Target:
#   Frozen Champion = BASE_PLUS_REGIME
#   Replay Year     = 2026
#
# PURPOSE:
#   Research Backtest
#          ==
#   Live Implementation
#
# IMPORTANT:
#   Production model is NOT used for historical predictions.
#
#   For replay year 2026 we reconstruct the exact historical
#   model using only information available before 2026.
#
#   NO optimization
#   NO parameter change
#   NO feature change
# ============================================================

import math
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 300)


# ============================================================
# 0. CONFIG
# ============================================================

REPLAY_YEAR = 2026

CHAMPION_NAME = "BASE_PLUS_REGIME"

FEATURE_TOLERANCE = 1e-10
PROBABILITY_TOLERANCE = 1e-10
POSITION_TOLERANCE = 1e-10
RETURN_TOLERANCE = 1e-12

# sequential live-feature checks
SEQUENTIAL_EXTRA_SAMPLES = 120

# 250 bars is comfortably larger than the longest feature
# lookback in the frozen 41-feature specification.
SEQUENTIAL_HISTORY_BARS = 250


# ============================================================
# 1. HARD PRECHECK
# ============================================================

REQUIRED_OBJECTS = [

    # data
    "bars",

    # tournament
    "FINAL_TOURNAMENT_TRADES",
    "FINAL_TOURNAMENT_ANNUAL",
    "FINAL_TOURNAMENT_FEATURE_SETS",
    "FINAL_TOURNAMENT_DECISION",

    # research pipeline
    "BASE_FEATURES",
    "BASE_COST",
    "make_final_tournament_features",
    "build_tournament_dataset",
    "make_split",
    "choose_calibration",
    "expanding_oof",
    "fit_hgb",
    "model_probability",
    "fit_calibrator",
    "prediction_frame",
    "choose_threshold_session",
    "select_trades",
    "choose_sizing",
    "stats_of_returns",

    # independent LIVE feature engine
    "make_live_features",
]


missing_objects = [

    name
    for name in REQUIRED_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Historical Replayに必要なNotebook状態が不足しています。\n\n"

        f"Missing:\n{missing_objects}\n\n"

        "Final Tournament → Champion Freeze → "
        "Live Inference Engine の順に実行してください。"
    )


if (

    FINAL_TOURNAMENT_DECISION
    !=
    "FREEZE_BASE_PLUS_REGIME"

):

    raise RuntimeError(

        "Frozen Champion decisionが一致しません。\n"

        f"Actual: {FINAL_TOURNAMENT_DECISION}"
    )


# ============================================================
# 2. CHAMPION FEATURES
# ============================================================

if (

    CHAMPION_NAME
    not in
    FINAL_TOURNAMENT_FEATURE_SETS

):

    raise RuntimeError(
        "Champion feature listがありません。"
    )


REPLAY_FEATURES = list(

    FINAL_TOURNAMENT_FEATURE_SETS[
        CHAMPION_NAME
    ]
)


if len(
    REPLAY_FEATURES
) != 41:

    raise RuntimeError(

        "Champion feature count != 41\n"

        f"Actual: {len(REPLAY_FEATURES)}"
    )


print("=" * 110)

print(
    "HISTORICAL REPLAY PARITY TEST"
)

print("=" * 110)

print(
    "Replay year:",
    REPLAY_YEAR
)

print(
    "Champion:",
    CHAMPION_NAME
)

print(
    "Features:",
    len(
        REPLAY_FEATURES
    )
)

print()

print(
    "IMPORTANT:"
)

print(
    "Production model is NOT used for historical replay."
)

print(
    "The historical walk-forward model is reconstructed"
)

print(
    "using only information available before the replay year."
)


# ============================================================
# 3. CLEAN BARS
# ============================================================

REPLAY_BARS = bars.copy()


REPLAY_BARS.columns = [

    str(c)
    .strip()
    .lower()

    for c in REPLAY_BARS.columns
]


required_ohlc = [

    "open",
    "high",
    "low",
    "close",
]


missing_ohlc = [

    c
    for c in required_ohlc

    if c not in REPLAY_BARS.columns
]


if missing_ohlc:

    raise RuntimeError(
        f"OHLC missing: {missing_ohlc}"
    )


REPLAY_BARS = (

    REPLAY_BARS[
        required_ohlc
    ]

    .copy()
)


if not isinstance(
    REPLAY_BARS.index,
    pd.DatetimeIndex
):

    raise RuntimeError(
        "bars.index is not DatetimeIndex."
    )


REPLAY_BARS.index = pd.to_datetime(

    REPLAY_BARS.index,

    utc=True,

    errors="coerce"
)


REPLAY_BARS = (

    REPLAY_BARS

    .loc[
        ~REPLAY_BARS.index.isna()
    ]

    .sort_index()
)


if (
    REPLAY_BARS.index
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate bar timestamps found."
    )


bad_grid = (

    (
        REPLAY_BARS.index.minute
        %
        15
    )
    !=
    0

    |

    (
        REPLAY_BARS.index.second
        !=
        0
    )

    |

    (
        REPLAY_BARS.index.microsecond
        !=
        0
    )
)


if np.asarray(
    bad_grid
).any():

    raise RuntimeError(
        "Non-15m timestamps found."
    )


print()

print(
    "Bars:",
    f"{len(REPLAY_BARS):,}"
)

print(
    "Period:",
    REPLAY_BARS.index.min(),
    "->",
    REPLAY_BARS.index.max()
)


# ============================================================
# 4. RESEARCH FEATURE FRAME
# ============================================================

RESEARCH_FEATURE_FRAME = (

    make_final_tournament_features(
        REPLAY_BARS
    )
)


missing_features = [

    feature
    for feature
    in REPLAY_FEATURES

    if feature not in
    RESEARCH_FEATURE_FRAME.columns
]


if missing_features:

    raise RuntimeError(

        "Research feature construction failed.\n"

        f"Missing: {missing_features}"
    )


# ============================================================
# 5. BUILD EXACT TOURNAMENT DATASET
#
# build_tournament_dataset uses notebook globals B and
# ALL_FEATURE_FRAME. Temporarily point them to clean replay data.
# ============================================================

OLD_B = globals().get(
    "B",
    None
)

OLD_ALL_FEATURE_FRAME = globals().get(
    "ALL_FEATURE_FRAME",
    None
)


globals()[
    "B"
] = REPLAY_BARS


globals()[
    "ALL_FEATURE_FRAME"
] = RESEARCH_FEATURE_FRAME


try:

    REPLAY_DATA = (

        build_tournament_dataset(
            REPLAY_FEATURES
        )
    )

finally:

    # Keep temporary values only until all objects are built.
    pass


print()

print("=" * 110)

print(
    "REPLAY DATASET"
)

print("=" * 110)

print(
    "Rows:",
    f"{len(REPLAY_DATA):,}"
)

print(
    "Signal period:",
    REPLAY_DATA.index.min(),
    "->",
    REPLAY_DATA.index.max()
)


# ============================================================
# 6. GET ORIGINAL FINAL-TOURNAMENT RESULT
# ============================================================

EXPECTED_ANNUAL_ROWS = (

    FINAL_TOURNAMENT_ANNUAL.loc[

        (
            FINAL_TOURNAMENT_ANNUAL[
                "feature_set"
            ]
            ==
            CHAMPION_NAME
        )

        &

        (
            FINAL_TOURNAMENT_ANNUAL[
                "test_year"
            ]
            ==
            REPLAY_YEAR
        )

    ]

    .copy()
)


if len(
    EXPECTED_ANNUAL_ROWS
) != 1:

    raise RuntimeError(

        "Expected Final Tournament annual rowが"
        "一意に取得できません。"
    )


EXPECTED_ANNUAL = (
    EXPECTED_ANNUAL_ROWS.iloc[0]
)


EXPECTED_TRADES = (

    FINAL_TOURNAMENT_TRADES.loc[

        (
            FINAL_TOURNAMENT_TRADES[
                "feature_set"
            ]
            ==
            CHAMPION_NAME
        )

        &

        (
            FINAL_TOURNAMENT_TRADES[
                "test_year"
            ]
            ==
            REPLAY_YEAR
        )

    ]

    .sort_index()

    .copy()
)


if len(
    EXPECTED_TRADES
) == 0:

    raise RuntimeError(
        "Expected tournament tradesがありません。"
    )


print()

print("=" * 110)

print(
    "EXPECTED FROZEN BACKTEST"
)

print("=" * 110)

print(
    "Calibration:",
    EXPECTED_ANNUAL[
        "calibration"
    ]
)

print(
    "Threshold:",
    EXPECTED_ANNUAL[
        "threshold"
    ]
)

print(
    "Session:",
    EXPECTED_ANNUAL[
        "session"
    ]
)

print(
    "Sizing:",
    EXPECTED_ANNUAL[
        "sizing"
    ]
)

print(
    "Trades:",
    len(
        EXPECTED_TRADES
    )
)

print(
    "PF:",
    EXPECTED_ANNUAL[
        "profit_factor"
    ]
)


# ============================================================
# 7. RECONSTRUCT HISTORICAL MODEL + POLICY
#
# Critical:
# We temporarily set BASE_FEATURES = 41 frozen Champion features.
#
# Nothing is tuned using 2026 result.
# Selection uses validation year 2025 exactly as before.
# ============================================================

ORIGINAL_BASE_FEATURES = list(
    BASE_FEATURES
)


try:

    globals()[
        "BASE_FEATURES"
    ] = list(
        REPLAY_FEATURES
    )


    SPLIT = make_split(

        REPLAY_DATA,

        REPLAY_YEAR
    )


    if SPLIT is None:

        raise RuntimeError(
            "Historical split failed."
        )


    HIST_TRAIN = SPLIT[
        "train"
    ]


    HIST_VALIDATION = SPLIT[
        "validation"
    ]


    HIST_FINAL_TRAIN = SPLIT[
        "final_train"
    ]


    HIST_TEST = SPLIT[
        "test"
    ]


    # --------------------------------------------------------
    # 7A. Calibration method selection
    # --------------------------------------------------------

    (
        REPLAY_CALIBRATION,

        REPLAY_CALIBRATION_TABLE

    ) = choose_calibration(

        HIST_TRAIN,

        HIST_VALIDATION
    )


    # --------------------------------------------------------
    # 7B. Validation model for threshold/session/sizing
    # --------------------------------------------------------

    HIST_TRAIN_OOF = expanding_oof(
        HIST_TRAIN
    )


    HIST_VALIDATION_MODEL = fit_hgb(
        HIST_TRAIN
    )


    HIST_VALIDATION_RAW = (

        model_probability(

            HIST_VALIDATION_MODEL,

            HIST_VALIDATION
        )
    )


    HIST_VALIDATION_CALIBRATOR = (

        fit_calibrator(

            REPLAY_CALIBRATION,

            HIST_TRAIN_OOF
        )
    )


    HIST_VALIDATION_CALIBRATED = np.clip(

        HIST_VALIDATION_CALIBRATOR.predict(
            HIST_VALIDATION_RAW
        ),

        0.0,

        1.0
    )


    HIST_VALIDATION_PREDICTION = (

        prediction_frame(

            HIST_VALIDATION,

            HIST_VALIDATION_CALIBRATED
        )
    )


    # --------------------------------------------------------
    # 7C. Threshold / session
    # --------------------------------------------------------

    (
        REPLAY_THRESHOLD,

        REPLAY_SESSION,

        REPLAY_THRESHOLD_TABLE

    ) = choose_threshold_session(

        HIST_VALIDATION_PREDICTION
    )


    HIST_VALIDATION_SELECTED = (

        select_trades(

            HIST_VALIDATION_PREDICTION,

            REPLAY_THRESHOLD,

            REPLAY_SESSION
        )
    )


    # --------------------------------------------------------
    # 7D. Sizing
    # --------------------------------------------------------

    (
        REPLAY_SIZING,

        REPLAY_SIZING_SCALE,

        REPLAY_SIZING_TABLE

    ) = choose_sizing(

        HIST_VALIDATION_SELECTED,

        REPLAY_THRESHOLD
    )


    # --------------------------------------------------------
    # 7E. Historical final model
    #
    # Uses ONLY data before replay year.
    # --------------------------------------------------------

    HIST_FINAL_OOF = expanding_oof(
        HIST_FINAL_TRAIN
    )


    HIST_FINAL_CALIBRATOR = (

        fit_calibrator(

            REPLAY_CALIBRATION,

            HIST_FINAL_OOF
        )
    )


    HIST_FINAL_MODEL = fit_hgb(
        HIST_FINAL_TRAIN
    )


finally:

    globals()[
        "BASE_FEATURES"
    ] = ORIGINAL_BASE_FEATURES


# ============================================================
# 8. POLICY REPRODUCTION CHECK
# ============================================================

POLICY_PARITY = {

    "calibration":

        str(
            REPLAY_CALIBRATION
        )

        ==

        str(
            EXPECTED_ANNUAL[
                "calibration"
            ]
        ),


    "threshold":

        np.isclose(

            float(
                REPLAY_THRESHOLD
            ),

            float(
                EXPECTED_ANNUAL[
                    "threshold"
                ]
            ),

            atol=1e-12,
            rtol=0.0
        ),


    "session":

        str(
            REPLAY_SESSION
        )

        ==

        str(
            EXPECTED_ANNUAL[
                "session"
            ]
        ),


    "sizing":

        str(
            REPLAY_SIZING
        )

        ==

        str(
            EXPECTED_ANNUAL[
                "sizing"
            ]
        ),
}


POLICY_PARITY_OK = all(
    POLICY_PARITY.values()
)


print()

print("=" * 110)

print(
    "HISTORICAL POLICY REPRODUCTION"
)

print("=" * 110)


for key, value in (
    POLICY_PARITY.items()
):

    print(
        f"{key}: {value}"
    )


print()

print(
    "POLICY PARITY PASSED:",
    POLICY_PARITY_OK
)


if not POLICY_PARITY_OK:

    raise RuntimeError(

        "Historical policy cannot reproduce Final Tournament.\n"

        "STOP before replay."
    )


# ============================================================
# 9. FULL-FRAME LIVE FEATURE ENGINE
#
# This is the independent deployment implementation.
# ============================================================

LIVE_FEATURE_FRAME_FULL = (

    make_live_features(
        REPLAY_BARS
    )
)


missing_live_features = [

    feature
    for feature
    in REPLAY_FEATURES

    if feature not in
    LIVE_FEATURE_FRAME_FULL.columns
]


if missing_live_features:

    raise RuntimeError(

        "Live feature engine missing features:\n"

        f"{missing_live_features}"
    )


# ============================================================
# 10. FEATURE PARITY - ALL REPLAY TEST ROWS
# ============================================================

TEST_INDEX = (
    HIST_TEST.index
)


research_test_features = (

    RESEARCH_FEATURE_FRAME

    .loc[
        TEST_INDEX,
        REPLAY_FEATURES
    ]

    .copy()
)


live_test_features = (

    LIVE_FEATURE_FRAME_FULL

    .loc[
        TEST_INDEX,
        REPLAY_FEATURES
    ]

    .copy()
)


if (
    research_test_features
    .isna()
    .any()
    .any()

    or

    live_test_features
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "Feature parity data contains NaN."
    )


FEATURE_ABS_DIFF = (

    research_test_features

    -

    live_test_features

).abs()


FEATURE_MAX_DIFF_BY_COLUMN = (

    FEATURE_ABS_DIFF

    .max()

    .sort_values(
        ascending=False
    )

    .rename(
        "max_abs_diff"
    )

    .reset_index()

    .rename(
        columns={
            "index":
                "feature"
        }
    )
)


FULL_FEATURE_MAX_DIFF = float(

    FEATURE_ABS_DIFF
    .to_numpy()
    .max()
)


FULL_FEATURE_PARITY_OK = bool(

    FULL_FEATURE_MAX_DIFF
    <=
    FEATURE_TOLERANCE
)


print()

print("=" * 110)

print(
    "FULL-FRAME FEATURE PARITY"
)

print("=" * 110)

print(
    "Rows checked:",
    f"{len(TEST_INDEX):,}"
)

print(
    "Features checked:",
    len(
        REPLAY_FEATURES
    )
)

print(
    "Max absolute difference:",
    FULL_FEATURE_MAX_DIFF
)

print(
    "Tolerance:",
    FEATURE_TOLERANCE
)

print(
    "PASSED:",
    FULL_FEATURE_PARITY_OK
)


# ============================================================
# 11. TRUE SEQUENTIAL FEATURE PARITY
#
# To verify that truncating data at time t gives exactly the
# same feature values as full-history calculation.
#
# We test:
#   - every expected trade signal
#   - extra evenly-spaced non-trade timestamps
# ============================================================

expected_trade_times = pd.DatetimeIndex(

    EXPECTED_TRADES.index
    .unique()
)


available_test_times = pd.DatetimeIndex(
    TEST_INDEX
)


if len(
    available_test_times
) == 0:

    raise RuntimeError(
        "No replay test timestamps."
    )


# ------------------------------------------------------------
# deterministic evenly-spaced samples
# ------------------------------------------------------------

sample_count = min(

    SEQUENTIAL_EXTRA_SAMPLES,

    len(
        available_test_times
    )
)


sample_positions = np.linspace(

    0,

    len(
        available_test_times
    )
    -
    1,

    num=sample_count,

    dtype=int
)


extra_sample_times = (

    available_test_times[
        sample_positions
    ]
)


SEQUENTIAL_TEST_TIMES = (

    expected_trade_times

    .union(
        extra_sample_times
    )

    .sort_values()
)


sequential_rows = []


print()

print("=" * 110)

print(
    "SEQUENTIAL FEATURE PARITY"
)

print("=" * 110)

print(
    "Replay timestamps:",
    len(
        SEQUENTIAL_TEST_TIMES
    )
)


for counter, signal_time in enumerate(

    SEQUENTIAL_TEST_TIMES,

    start=1
):

    history = (

        REPLAY_BARS

        .loc[
            :
            signal_time
        ]

        .tail(
            SEQUENTIAL_HISTORY_BARS
        )

        .copy()
    )


    if len(
        history
    ) < 150:

        continue


    sequential_features = (

        make_live_features(
            history
        )
    )


    live_row = (

        sequential_features

        .loc[
            signal_time,
            REPLAY_FEATURES
        ]
    )


    research_row = (

        RESEARCH_FEATURE_FRAME

        .loc[
            signal_time,
            REPLAY_FEATURES
        ]
    )


    diff = (

        live_row

        -

        research_row

    ).abs()


    max_diff = float(
        diff.max()
    )


    sequential_rows.append(

        {
            "signal_time":
                signal_time,

            "max_feature_diff":
                max_diff,

            "passed":
                bool(
                    max_diff
                    <=
                    FEATURE_TOLERANCE
                ),
        }
    )


    if (
        counter % 100
        ==
        0
    ):

        print(
            f"Checked {counter:,} / "
            f"{len(SEQUENTIAL_TEST_TIMES):,}"
        )


SEQUENTIAL_FEATURE_PARITY = pd.DataFrame(
    sequential_rows
)


SEQUENTIAL_FEATURE_PARITY_OK = bool(

    len(
        SEQUENTIAL_FEATURE_PARITY
    )
    >
    0

    and

    SEQUENTIAL_FEATURE_PARITY[
        "passed"
    ].all()
)


SEQUENTIAL_MAX_DIFF = float(

    SEQUENTIAL_FEATURE_PARITY[
        "max_feature_diff"
    ].max()
)


print()

print(
    "Sequential rows checked:",
    len(
        SEQUENTIAL_FEATURE_PARITY
    )
)

print(
    "Max sequential feature diff:",
    SEQUENTIAL_MAX_DIFF
)

print(
    "PASSED:",
    SEQUENTIAL_FEATURE_PARITY_OK
)


# ============================================================
# 12. BACKTEST PROBABILITIES USING RESEARCH FEATURES
# ============================================================

research_raw_probability = (

    HIST_FINAL_MODEL

    .predict_proba(

        research_test_features

    )[:, 1]
)


research_calibrated_probability = np.clip(

    HIST_FINAL_CALIBRATOR.predict(
        research_raw_probability
    ),

    0.0,

    1.0
)


# ============================================================
# 13. SAME HISTORICAL MODEL USING LIVE FEATURES
#
# This directly tests deployment feature compatibility.
# ============================================================

live_raw_probability = (

    HIST_FINAL_MODEL

    .predict_proba(

        live_test_features

    )[:, 1]
)


live_calibrated_probability = np.clip(

    HIST_FINAL_CALIBRATOR.predict(
        live_raw_probability
    ),

    0.0,

    1.0
)


RAW_PROB_DIFF = np.abs(

    research_raw_probability

    -

    live_raw_probability
)


CAL_PROB_DIFF = np.abs(

    research_calibrated_probability

    -

    live_calibrated_probability
)


MAX_RAW_PROB_DIFF = float(
    RAW_PROB_DIFF.max()
)


MAX_CAL_PROB_DIFF = float(
    CAL_PROB_DIFF.max()
)


PROBABILITY_PARITY_OK = bool(

    MAX_RAW_PROB_DIFF
    <=
    PROBABILITY_TOLERANCE

    and

    MAX_CAL_PROB_DIFF
    <=
    PROBABILITY_TOLERANCE
)


print()

print("=" * 110)

print(
    "PROBABILITY PARITY"
)

print("=" * 110)

print(
    "Rows checked:",
    f"{len(TEST_INDEX):,}"
)

print(
    "Max RAW p_up diff:",
    MAX_RAW_PROB_DIFF
)

print(
    "Max calibrated p_up diff:",
    MAX_CAL_PROB_DIFF
)

print(
    "Tolerance:",
    PROBABILITY_TOLERANCE
)

print(
    "PASSED:",
    PROBABILITY_PARITY_OK
)


# ============================================================
# 14. BUILD LIVE-IMPLEMENTATION PREDICTION FRAME
#
# Uses LIVE feature probabilities, not research feature values.
# ============================================================

LIVE_REPLAY_PREDICTION = (

    prediction_frame(

        HIST_TEST,

        live_calibrated_probability
    )
)


# ============================================================
# 15. INDEPENDENT SESSION CHECK
# ============================================================

def replay_session_allowed(
    timestamp,
    policy
):

    timestamp = pd.Timestamp(
        timestamp
    )


    if timestamp.tzinfo is None:

        timestamp = timestamp.tz_localize(
            "UTC"
        )

    else:

        timestamp = timestamp.tz_convert(
            "UTC"
        )


    hour = int(
        timestamp.hour
    )


    if policy == "ALL":

        return True


    if policy == "UTC_13_24":

        return (
            hour
            >=
            13
        )


    if policy == "UTC_21_24":

        return (
            hour
            >=
            21
        )


    if policy == "EXCLUDE_08_13":

        return not (

            8
            <=
            hour
            <
            13
        )


    raise RuntimeError(
        f"Unknown session: {policy}"
    )


# ============================================================
# 16. INDEPENDENT POSITION SIZE
#
# Mirrors frozen deployment logic independently from
# apply_sizing().
# ============================================================

def replay_position_size(
    confidence,
    threshold,
    policy,
    scale
):

    confidence = float(
        confidence
    )


    edge = (

        confidence
        -
        threshold

    ) / max(

        1.0
        -
        threshold,

        1e-12
    )


    edge = float(

        np.clip(

            edge,

            0.0,

            1.0
        )
    )


    if policy == "FIXED":

        raw_size = 1.0


    elif policy == "GENTLE":

        raw_size = (

            0.85
            +
            0.30
            *
            edge
        )


    elif policy == "MODERATE":

        raw_size = (

            0.70
            +
            0.60
            *
            edge
        )


    elif policy == "STRONG":

        raw_size = (

            0.50
            +
            1.00
            *
            edge
        )


    else:

        raise RuntimeError(
            f"Unknown sizing policy: {policy}"
        )


    return float(

        np.clip(

            raw_size
            *
            scale,

            0.25,

            2.0
        )
    )


# ============================================================
# 17. TRUE SEQUENTIAL SIGNAL / POSITION REPLAY
#
# We simulate the live position state:
#
# signal t
#   -> entry t+15
#   -> exit  t+45
#
# A new trade is blocked while entry_time < existing exit.
# ============================================================

replay_trade_rows = []

open_until = None


for signal_time, row in (
    LIVE_REPLAY_PREDICTION
    .sort_index()
    .iterrows()
):

    confidence = float(
        row[
            "confidence"
        ]
    )


    threshold_allowed = bool(

        confidence

        >=

        REPLAY_THRESHOLD
    )


    session_allowed = (

        replay_session_allowed(

            signal_time,

            REPLAY_SESSION
        )
    )


    entry_time = row[
        "entry_time"
    ]


    label_end = row[
        "label_end"
    ]


    position_blocked = bool(

        open_until is not None

        and

        entry_time
        <
        open_until
    )


    if not session_allowed:

        action = (
            "NO_TRADE"
        )

        reason = (
            "SESSION_BLOCKED"
        )


    elif not threshold_allowed:

        action = (
            "NO_TRADE"
        )

        reason = (
            "LOW_CONFIDENCE"
        )


    elif position_blocked:

        action = (
            "NO_TRADE"
        )

        reason = (
            "OPEN_POSITION_OVERLAP_BLOCKED"
        )


    else:

        action = str(
            row[
                "side"
            ]
        )

        reason = (
            "TRADE_SIGNAL"
        )


    if action in [
        "BUY",
        "SELL",
    ]:

        position_size = (

            replay_position_size(

                confidence,

                REPLAY_THRESHOLD,

                REPLAY_SIZING,

                REPLAY_SIZING_SCALE
            )
        )


        net_return = (

            position_size

            *

            (

                float(
                    row[
                        "gross_return"
                    ]
                )

                -

                float(
                    BASE_COST
                )
            )
        )


        replay_trade_rows.append(

            {
                "signal_time":
                    signal_time,

                "side":
                    action,

                "confidence":
                    confidence,

                "p_up":
                    float(
                        row[
                            "p_up"
                        ]
                    ),

                "position_size":
                    position_size,

                "entry_time":
                    entry_time,

                "exit_time":
                    label_end,

                "gross_return":
                    float(
                        row[
                            "gross_return"
                        ]
                    ),

                "net_return":
                    net_return,

                "reason":
                    reason,
            }
        )


        open_until = (
            label_end
        )


REPLAY_LIVE_TRADES = (

    pd.DataFrame(
        replay_trade_rows
    )

    .set_index(
        "signal_time"
    )

    .sort_index()
)


# ============================================================
# 18. TRADE TIMESTAMP PARITY
# ============================================================

expected_index = pd.DatetimeIndex(

    EXPECTED_TRADES.index
)


actual_index = pd.DatetimeIndex(

    REPLAY_LIVE_TRADES.index
)


missing_trade_times = (

    expected_index

    .difference(
        actual_index
    )
)


unexpected_trade_times = (

    actual_index

    .difference(
        expected_index
    )
)


TRADE_TIMESTAMP_PARITY_OK = bool(

    len(
        missing_trade_times
    )
    ==
    0

    and

    len(
        unexpected_trade_times
    )
    ==
    0

    and

    len(
        actual_index
    )
    ==
    len(
        expected_index
    )
)


print()

print("=" * 110)

print(
    "TRADE TIMESTAMP PARITY"
)

print("=" * 110)

print(
    "Expected trades:",
    len(
        expected_index
    )
)

print(
    "Replay trades:",
    len(
        actual_index
    )
)

print(
    "Missing trades:",
    len(
        missing_trade_times
    )
)

print(
    "Unexpected trades:",
    len(
        unexpected_trade_times
    )
)

print(
    "PASSED:",
    TRADE_TIMESTAMP_PARITY_OK
)


# ============================================================
# 19. MATCHED TRADE DETAIL PARITY
# ============================================================

common_index = (

    expected_index

    .intersection(
        actual_index
    )
)


EXPECTED_MATCHED = (

    EXPECTED_TRADES

    .loc[
        common_index
    ]

    .copy()
)


ACTUAL_MATCHED = (

    REPLAY_LIVE_TRADES

    .loc[
        common_index
    ]

    .copy()
)


detail_rows = []


for signal_time in (
    common_index
):

    expected = (
        EXPECTED_MATCHED
        .loc[
            signal_time
        ]
    )


    actual = (
        ACTUAL_MATCHED
        .loc[
            signal_time
        ]
    )


    # In the unlikely case duplicated DataFrame rows appear
    if isinstance(
        expected,
        pd.DataFrame
    ):

        expected = (
            expected.iloc[0]
        )


    if isinstance(
        actual,
        pd.DataFrame
    ):

        actual = (
            actual.iloc[0]
        )


    side_equal = (

        str(
            expected[
                "side"
            ]
        )

        ==

        str(
            actual[
                "side"
            ]
        )
    )


    confidence_diff = abs(

        float(
            expected[
                "confidence"
            ]
        )

        -

        float(
            actual[
                "confidence"
            ]
        )
    )


    position_diff = abs(

        float(
            expected[
                "position_size"
            ]
        )

        -

        float(
            actual[
                "position_size"
            ]
        )
    )


    net_return_diff = abs(

        float(
            expected[
                "net_return"
            ]
        )

        -

        float(
            actual[
                "net_return"
            ]
        )
    )


    entry_equal = True


    if "entry_time" in EXPECTED_MATCHED.columns:

        entry_equal = (

            pd.Timestamp(
                expected[
                    "entry_time"
                ]
            )

            ==

            pd.Timestamp(
                actual[
                    "entry_time"
                ]
            )
        )


    exit_equal = True


    if "label_end" in EXPECTED_MATCHED.columns:

        exit_equal = (

            pd.Timestamp(
                expected[
                    "label_end"
                ]
            )

            ==

            pd.Timestamp(
                actual[
                    "exit_time"
                ]
            )
        )


    passed = bool(

        side_equal

        and

        confidence_diff
        <=
        PROBABILITY_TOLERANCE

        and

        position_diff
        <=
        POSITION_TOLERANCE

        and

        net_return_diff
        <=
        RETURN_TOLERANCE

        and

        entry_equal

        and

        exit_equal
    )


    detail_rows.append(

        {
            "signal_time":
                signal_time,

            "side_equal":
                side_equal,

            "confidence_diff":
                confidence_diff,

            "position_size_diff":
                position_diff,

            "net_return_diff":
                net_return_diff,

            "entry_time_equal":
                entry_equal,

            "exit_time_equal":
                exit_equal,

            "passed":
                passed,
        }
    )


TRADE_DETAIL_PARITY = pd.DataFrame(
    detail_rows
)


TRADE_DETAIL_PARITY_OK = bool(

    len(
        TRADE_DETAIL_PARITY
    )
    ==
    len(
        EXPECTED_TRADES
    )

    and

    TRADE_DETAIL_PARITY[
        "passed"
    ].all()
)


# ============================================================
# 20. DUPLICATE / OVERLAP CHECK
# ============================================================

REPLAY_DUPLICATE_SIGNALS = int(

    REPLAY_LIVE_TRADES.index
    .duplicated()
    .sum()
)


REPLAY_OVERLAPS = 0


if len(
    REPLAY_LIVE_TRADES
) > 1:

    replay_sorted = (

        REPLAY_LIVE_TRADES
        .sort_values(
            "entry_time"
        )
    )


    previous_exit = (

        replay_sorted[
            "exit_time"
        ]
        .shift(
            1
        )
    )


    REPLAY_OVERLAPS = int(

        (

            replay_sorted[
                "entry_time"
            ]

            <

            previous_exit
        )

        .fillna(
            False
        )

        .sum()
    )


INTEGRITY_OK = bool(

    REPLAY_DUPLICATE_SIGNALS
    ==
    0

    and

    REPLAY_OVERLAPS
    ==
    0
)


# ============================================================
# 21. PERFORMANCE REPRODUCTION
#
# This is not a new performance test.
# It only verifies that the replay trade list reproduces
# the already-frozen historical result.
# ============================================================

REPLAY_STATS = stats_of_returns(

    REPLAY_LIVE_TRADES[
        "net_return"
    ]
)


EXPECTED_STATS = stats_of_returns(

    EXPECTED_TRADES[
        "net_return"
    ]
)


PERFORMANCE_PARITY = {

    "trades":

        REPLAY_STATS[
            "trades"
        ]

        ==

        EXPECTED_STATS[
            "trades"
        ],


    "avg_return":

        np.isclose(

            REPLAY_STATS[
                "avg_return"
            ],

            EXPECTED_STATS[
                "avg_return"
            ],

            atol=RETURN_TOLERANCE,
            rtol=0.0
        ),


    "profit_factor":

        np.isclose(

            REPLAY_STATS[
                "profit_factor"
            ],

            EXPECTED_STATS[
                "profit_factor"
            ],

            atol=1e-10,
            rtol=0.0
        ),


    "growth":

        np.isclose(

            REPLAY_STATS[
                "growth"
            ],

            EXPECTED_STATS[
                "growth"
            ],

            atol=1e-10,
            rtol=0.0
        ),


    "max_dd":

        np.isclose(

            REPLAY_STATS[
                "max_dd"
            ],

            EXPECTED_STATS[
                "max_dd"
            ],

            atol=1e-10,
            rtol=0.0
        ),
}


PERFORMANCE_PARITY_OK = all(
    PERFORMANCE_PARITY.values()
)


# ============================================================
# 22. FINAL DECISION
# ============================================================

FINAL_CHECKS = {

    "historical_policy":
        POLICY_PARITY_OK,

    "full_frame_features":
        FULL_FEATURE_PARITY_OK,

    "sequential_features":
        SEQUENTIAL_FEATURE_PARITY_OK,

    "probabilities":
        PROBABILITY_PARITY_OK,

    "trade_timestamps":
        TRADE_TIMESTAMP_PARITY_OK,

    "trade_details":
        TRADE_DETAIL_PARITY_OK,

    "integrity":
        INTEGRITY_OK,

    "performance_reproduction":
        PERFORMANCE_PARITY_OK,
}


HISTORICAL_REPLAY_PARITY_OK = all(

    FINAL_CHECKS.values()
)


if (
    HISTORICAL_REPLAY_PARITY_OK
):

    HISTORICAL_REPLAY_DECISION = (

        "PASS_READY_FOR_PAPER_EXECUTION_ENGINE"
    )


else:

    HISTORICAL_REPLAY_DECISION = (

        "FAIL_DO_NOT_START_PAPER_TRADING"
    )


# ============================================================
# 23. DISPLAY HELPER
# ============================================================

def replay_show(
    title,
    frame,
    max_rows=30
):

    print()

    print("=" * 110)

    print(
        title
    )

    print("=" * 110)


    if frame is None or len(
        frame
    ) == 0:

        print(
            "No data"
        )

        return


    view = frame.head(
        max_rows
    )


    try:

        display(
            view
        )

    except Exception:

        print(
            view.to_string()
        )


# ============================================================
# 24. OUTPUT TABLES
# ============================================================

replay_show(

    "TOP FEATURE DIFFERENCES",

    FEATURE_MAX_DIFF_BY_COLUMN,

    max_rows=41
)


if not (
    SEQUENTIAL_FEATURE_PARITY[
        "passed"
    ].all()
):

    replay_show(

        "FAILED SEQUENTIAL FEATURE ROWS",

        SEQUENTIAL_FEATURE_PARITY.loc[

            SEQUENTIAL_FEATURE_PARITY[
                "passed"
            ]
            ==
            False

        ],

        max_rows=50
    )


if len(
    missing_trade_times
) > 0:

    replay_show(

        "MISSING TRADE TIMES",

        pd.DataFrame(

            {
                "signal_time":
                    missing_trade_times
            }
        ),

        max_rows=50
    )


if len(
    unexpected_trade_times
) > 0:

    replay_show(

        "UNEXPECTED TRADE TIMES",

        pd.DataFrame(

            {
                "signal_time":
                    unexpected_trade_times
            }
        ),

        max_rows=50
    )


if len(
    TRADE_DETAIL_PARITY
):

    failed_details = (

        TRADE_DETAIL_PARITY.loc[

            TRADE_DETAIL_PARITY[
                "passed"
            ]
            ==
            False

        ]

        .copy()
    )


    if len(
        failed_details
    ):

        replay_show(

            "FAILED TRADE DETAIL PARITY",

            failed_details,

            max_rows=50
        )


# ============================================================
# 25. FINAL REPORT
# ============================================================

print()

print("=" * 110)

print(
    "HISTORICAL REPLAY FINAL REPORT"
)

print("=" * 110)


print(
    "Replay year:",
    REPLAY_YEAR
)

print(
    "Champion:",
    CHAMPION_NAME
)


print()

print(
    "Expected trades:",
    len(
        EXPECTED_TRADES
    )
)

print(
    "Replay trades:",
    len(
        REPLAY_LIVE_TRADES
    )
)


print()

print(
    "Max full-frame feature diff:",
    FULL_FEATURE_MAX_DIFF
)

print(
    "Max sequential feature diff:",
    SEQUENTIAL_MAX_DIFF
)

print(
    "Max RAW probability diff:",
    MAX_RAW_PROB_DIFF
)

print(
    "Max calibrated probability diff:",
    MAX_CAL_PROB_DIFF
)


print()

print(
    "Missing trades:",
    len(
        missing_trade_times
    )
)

print(
    "Unexpected trades:",
    len(
        unexpected_trade_times
    )
)

print(
    "Duplicate trades:",
    REPLAY_DUPLICATE_SIGNALS
)

print(
    "Overlapping trades:",
    REPLAY_OVERLAPS
)


print()

print(
    "EXPECTED PERFORMANCE"
)

print(
    "PF:",
    EXPECTED_STATS[
        "profit_factor"
    ]
)

print(
    "Avg:",
    EXPECTED_STATS[
        "avg_return"
    ]
)

print(
    "Growth:",
    EXPECTED_STATS[
        "growth"
    ]
)

print(
    "Max DD:",
    EXPECTED_STATS[
        "max_dd"
    ]
)


print()

print(
    "REPLAY PERFORMANCE"
)

print(
    "PF:",
    REPLAY_STATS[
        "profit_factor"
    ]
)

print(
    "Avg:",
    REPLAY_STATS[
        "avg_return"
    ]
)

print(
    "Growth:",
    REPLAY_STATS[
        "growth"
    ]
)

print(
    "Max DD:",
    REPLAY_STATS[
        "max_dd"
    ]
)


print()

print("=" * 110)

print(
    "FINAL CHECKS"
)

print("=" * 110)


for check_name, passed in (
    FINAL_CHECKS.items()
):

    print(
        f"{check_name}: {passed}"
    )


print()

print(
    "HISTORICAL REPLAY PARITY PASSED:",
    HISTORICAL_REPLAY_PARITY_OK
)


print()

print(
    "FINAL DECISION:",
    HISTORICAL_REPLAY_DECISION
)


if (
    HISTORICAL_REPLAY_PARITY_OK
):

    print()

    print(
        "Research backtest and deployment logic are consistent."
    )

    print(
        "Next step: PAPER EXECUTION ENGINE."
    )

    print(
        "No further model/feature optimization is required."
    )


else:

    print()

    print(
        "STOP."
    )

    print(
        "Do not start Paper Trading until every failed parity"
    )

    print(
        "check has been diagnosed and corrected."
    )


# ============================================================
# 26. SAVE NOTEBOOK VARIABLES
# ============================================================

HISTORICAL_REPLAY_YEAR = (
    REPLAY_YEAR
)

HISTORICAL_REPLAY_EXPECTED_TRADES = (
    EXPECTED_TRADES.copy()
)

HISTORICAL_REPLAY_LIVE_TRADES = (
    REPLAY_LIVE_TRADES.copy()
)

HISTORICAL_REPLAY_FEATURE_DIFF = (
    FEATURE_MAX_DIFF_BY_COLUMN.copy()
)

HISTORICAL_REPLAY_SEQUENTIAL_FEATURE_PARITY = (
    SEQUENTIAL_FEATURE_PARITY.copy()
)

HISTORICAL_REPLAY_TRADE_DETAIL_PARITY = (
    TRADE_DETAIL_PARITY.copy()
)

HISTORICAL_REPLAY_FINAL_CHECKS = (
    FINAL_CHECKS.copy()
)

HISTORICAL_REPLAY_PARITY_OK = (
    HISTORICAL_REPLAY_PARITY_OK
)

HISTORICAL_REPLAY_DECISION = (
    HISTORICAL_REPLAY_DECISION
)


# ============================================================
# 27. RESTORE NOTEBOOK GLOBALS
# ============================================================

globals()[
    "BASE_FEATURES"
] = ORIGINAL_BASE_FEATURES


if OLD_B is not None:

    globals()[
        "B"
    ] = OLD_B


if OLD_ALL_FEATURE_FRAME is not None:

    globals()[
        "ALL_FEATURE_FRAME"
    ] = OLD_ALL_FEATURE_FRAME


print()

print("=" * 110)

print(
    "REPLAY TEST COMPLETE"
)

print("=" * 110)

print(
    "Saved variables:"
)

print(
    "HISTORICAL_REPLAY_LIVE_TRADES"
)

print(
    "HISTORICAL_REPLAY_FEATURE_DIFF"
)

print(
    "HISTORICAL_REPLAY_TRADE_DETAIL_PARITY"
)

print(
    "HISTORICAL_REPLAY_FINAL_CHECKS"
)

print(
    "HISTORICAL_REPLAY_DECISION"
)


## 元セルindex 71
構文状態：valid


In [ ]:
# ============================================================
# SEQUENTIAL PARITY FAILURE - EXACT DIAGNOSTIC
#
# Purpose:
#   2026-06-11 02:45 UTC の1件だけ発生した
#   max_feature_diff = 1/3 の正体を特定する。
#
# No training
# No optimization
# No strategy modification
# ============================================================

import numpy as np
import pandas as pd

FAILED_TIME = pd.Timestamp(
    "2026-06-11 02:45:00",
    tz="UTC"
)

HISTORY_LENGTHS = [
    150,
    200,
    250,
    300,
    500,
    1000,
]


# ============================================================
# 1. PRECHECK
# ============================================================

required = [
    "REPLAY_BARS",
    "RESEARCH_FEATURE_FRAME",
    "REPLAY_FEATURES",
    "make_live_features",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Required objects missing: {missing}\n"
        "Historical Replay cellを先に実行してください。"
    )


if FAILED_TIME not in RESEARCH_FEATURE_FRAME.index:
    raise RuntimeError(
        f"{FAILED_TIME} not found in research feature frame."
    )


# ============================================================
# 2. RESEARCH REFERENCE
# ============================================================

reference = (
    RESEARCH_FEATURE_FRAME
    .loc[
        FAILED_TIME,
        REPLAY_FEATURES
    ]
    .astype(float)
)


# ============================================================
# 3. EXACT 250-BAR SEQUENTIAL RESULT
# ============================================================

history_250 = (
    REPLAY_BARS
    .loc[:FAILED_TIME]
    .tail(250)
    .copy()
)

live_250_frame = make_live_features(
    history_250
)

live_250 = (
    live_250_frame
    .loc[
        FAILED_TIME,
        REPLAY_FEATURES
    ]
    .astype(float)
)


comparison = pd.DataFrame({
    "research": reference,
    "sequential_250": live_250,
})

comparison["diff"] = (
    comparison["sequential_250"]
    -
    comparison["research"]
)

comparison["abs_diff"] = (
    comparison["diff"].abs()
)

comparison = comparison.sort_values(
    "abs_diff",
    ascending=False
)


print("=" * 100)
print("FAILED TIMESTAMP FEATURE DIAGNOSTIC")
print("=" * 100)

print("Timestamp:", FAILED_TIME)
print("History rows:", len(history_250))
print()

display(
    comparison.loc[
        comparison["abs_diff"] > 1e-12
    ]
)


# ============================================================
# 4. TEST DIFFERENT HISTORY LENGTHS
# ============================================================

length_rows = []

for n in HISTORY_LENGTHS:

    history = (
        REPLAY_BARS
        .loc[:FAILED_TIME]
        .tail(n)
        .copy()
    )

    if len(history) < 150:
        continue

    lf = make_live_features(
        history
    )

    row = (
        lf
        .loc[
            FAILED_TIME,
            REPLAY_FEATURES
        ]
        .astype(float)
    )

    diff = (
        row
        -
        reference
    ).abs()

    worst_feature = diff.idxmax()
    max_diff = float(diff.max())

    length_rows.append({
        "history_bars": n,
        "actual_rows": len(history),
        "worst_feature": worst_feature,
        "max_abs_diff": max_diff,
        "passed_1e10": bool(max_diff <= 1e-10),
    })


HISTORY_LENGTH_DIAGNOSTIC = pd.DataFrame(
    length_rows
)


print()
print("=" * 100)
print("HISTORY LENGTH SENSITIVITY")
print("=" * 100)

display(
    HISTORY_LENGTH_DIAGNOSTIC
)


# ============================================================
# 5. INVESTIGATE ALIGNMENT FEATURES
# ============================================================

alignment_features = [
    "ma20_slope",
    "ma50_slope",
    "ma100_slope",
    "ma_alignment_score",
    "slope_alignment_score",
    "directional_persistence_16",
]

alignment_features = [
    f for f in alignment_features
    if f in REPLAY_FEATURES
]


alignment_compare = pd.DataFrame({
    "research":
        reference[
            alignment_features
        ],

    "sequential_250":
        live_250[
            alignment_features
        ],
})

alignment_compare["diff"] = (
    alignment_compare["sequential_250"]
    -
    alignment_compare["research"]
)


print()
print("=" * 100)
print("ALIGNMENT / SLOPE FEATURES")
print("=" * 100)

display(
    alignment_compare
)


# ============================================================
# 6. DIRECT MA CALCULATION AROUND FAILED POINT
# ============================================================

full = (
    REPLAY_BARS
    .loc[:FAILED_TIME]
    .copy()
)

short = (
    full
    .tail(250)
    .copy()
)


ma_rows = []

for p in [20, 50, 100]:

    full_ma = (
        full["close"]
        .rolling(p)
        .mean()
    )

    short_ma = (
        short["close"]
        .rolling(p)
        .mean()
    )

    full_now = float(
        full_ma.loc[FAILED_TIME]
    )

    short_now = float(
        short_ma.loc[FAILED_TIME]
    )

    previous_time = (
        full.index[
            full.index.get_loc(FAILED_TIME) - 1
        ]
    )

    full_previous = float(
        full_ma.loc[previous_time]
    )

    short_previous = float(
        short_ma.loc[previous_time]
    )

    full_slope = (
        full_now
        /
        full_previous
        -
        1.0
    )

    short_slope = (
        short_now
        /
        short_previous
        -
        1.0
    )

    ma_rows.append({
        "period": p,

        "full_ma":
            full_now,

        "short_ma":
            short_now,

        "ma_diff":
            short_now - full_now,

        "full_prev_ma":
            full_previous,

        "short_prev_ma":
            short_previous,

        "prev_ma_diff":
            short_previous - full_previous,

        "full_slope":
            full_slope,

        "short_slope":
            short_slope,

        "slope_diff":
            short_slope - full_slope,

        "full_sign":
            np.sign(full_slope),

        "short_sign":
            np.sign(short_slope),
    })


MA_SLOPE_DIAGNOSTIC = pd.DataFrame(
    ma_rows
)


print()
print("=" * 100)
print("RAW MOVING-AVERAGE SLOPE DIAGNOSTIC")
print("=" * 100)

display(
    MA_SLOPE_DIAGNOSTIC
)


# ============================================================
# 7. FINAL DIAGNOSIS
# ============================================================

failed = comparison.loc[
    comparison["abs_diff"] > 1e-12
].copy()


print()
print("=" * 100)
print("DIAGNOSTIC SUMMARY")
print("=" * 100)

print(
    "Number of differing features:",
    len(failed)
)

if len(failed):

    print(
        "Worst feature:",
        failed.index[0]
    )

    print(
        "Worst absolute difference:",
        failed.iloc[0]["abs_diff"]
    )

    print()
    print("Differing feature names:")

    for feature in failed.index:
        print(
            " -",
            feature
        )

else:

    print(
        "No feature difference reproduced."
    )


print()
print(
    "Do NOT change the strategy yet."
)

print(
    "Next action depends on the exact feature shown above."
)


## 元セルindex 72
構文状態：valid


In [ ]:
# ============================================================
# FULL-HISTORY PREFIX PARITY FIX TEST
#
# Purpose:
#   tail(250) をやめて、
#   実運用と同じ「canonical historyを全部保持する方式」で
#   Sequential Feature Parityを再確認する。
#
# NO training
# NO optimization
# NO strategy change
# ============================================================

import numpy as np
import pandas as pd

FEATURE_TOLERANCE = 1e-10


# ------------------------------------------------------------
# PRECHECK
# ------------------------------------------------------------

required = [
    "REPLAY_BARS",
    "RESEARCH_FEATURE_FRAME",
    "REPLAY_FEATURES",
    "SEQUENTIAL_TEST_TIMES",
    "make_live_features",
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}\n"
        "前のHistorical Replayセルを先に実行してください。"
    )


# ------------------------------------------------------------
# FULL PREFIX REPLAY
# ------------------------------------------------------------

rows = []


print("=" * 100)
print("FULL-HISTORY PREFIX SEQUENTIAL PARITY")
print("=" * 100)

print(
    "Timestamps to check:",
    len(SEQUENTIAL_TEST_TIMES)
)

print(
    "Mode: ALL canonical historical bars up to each timestamp"
)

print()


for i, signal_time in enumerate(
    SEQUENTIAL_TEST_TIMES,
    start=1
):

    # ========================================================
    # IMPORTANT:
    # tail(250) は使わない。
    #
    # canonical datasetの開始からsignal_timeまで全部渡す。
    # ========================================================

    history = (
        REPLAY_BARS
        .loc[:signal_time]
        .copy()
    )


    live_frame = make_live_features(
        history
    )


    live_row = (
        live_frame
        .loc[
            signal_time,
            REPLAY_FEATURES
        ]
        .astype(float)
    )


    research_row = (
        RESEARCH_FEATURE_FRAME
        .loc[
            signal_time,
            REPLAY_FEATURES
        ]
        .astype(float)
    )


    diff = (
        live_row
        -
        research_row
    ).abs()


    max_diff = float(
        diff.max()
    )


    worst_feature = (
        diff.idxmax()
    )


    passed = bool(
        max_diff
        <=
        FEATURE_TOLERANCE
    )


    rows.append({
        "signal_time":
            signal_time,

        "history_rows":
            len(history),

        "worst_feature":
            worst_feature,

        "max_feature_diff":
            max_diff,

        "passed":
            passed,
    })


    if i % 100 == 0:

        print(
            f"Checked {i:,} / "
            f"{len(SEQUENTIAL_TEST_TIMES):,}"
        )


FULL_PREFIX_PARITY = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

failed = (
    FULL_PREFIX_PARITY
    .loc[
        FULL_PREFIX_PARITY[
            "passed"
        ]
        ==
        False
    ]
    .copy()
)


FULL_PREFIX_PARITY_OK = bool(
    len(failed) == 0
)


max_diff_all = float(
    FULL_PREFIX_PARITY[
        "max_feature_diff"
    ].max()
)


print()
print("=" * 100)
print("RESULT")
print("=" * 100)

print(
    "Rows checked:",
    len(FULL_PREFIX_PARITY)
)

print(
    "Passed:",
    int(
        FULL_PREFIX_PARITY[
            "passed"
        ].sum()
    )
)

print(
    "Failed:",
    len(failed)
)

print(
    "Maximum feature difference:",
    max_diff_all
)

print()

print(
    "FULL PREFIX PARITY PASSED:",
    FULL_PREFIX_PARITY_OK
)


if len(failed):

    print()
    print("=" * 100)
    print("FAILED ROWS")
    print("=" * 100)

    display(
        failed.head(50)
    )


# ------------------------------------------------------------
# ORIGINAL FAILED TIMESTAMP
# ------------------------------------------------------------

FAILED_TIME = pd.Timestamp(
    "2026-06-11 02:45:00",
    tz="UTC"
)


specific = (
    FULL_PREFIX_PARITY
    .loc[
        FULL_PREFIX_PARITY[
            "signal_time"
        ]
        ==
        FAILED_TIME
    ]
)


print()
print("=" * 100)
print("ORIGINAL FAILURE TIMESTAMP")
print("=" * 100)

if len(specific):

    display(
        specific
    )

else:

    print(
        "Timestamp was not part of replay sample."
    )


# ------------------------------------------------------------
# FINAL DECISION
# ------------------------------------------------------------

print()
print("=" * 100)
print("FINAL DECISION")
print("=" * 100)


if FULL_PREFIX_PARITY_OK:

    print(
        "PASS_CANONICAL_FULL_HISTORY_SOLVES_PARITY"
    )

    print()

    print(
        "The frozen strategy does NOT need to be changed."
    )

    print(
        "Production rule:"
    )

    print(
        "Never calculate live features from an arbitrary "
        "tail(N) history."
    )

    print(
        "Maintain the canonical clean historical series and "
        "append each newly closed 15m bar."
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "Harden Live Inference Engine to enforce canonical "
        "history, then proceed to Paper Execution Engine."
    )

else:

    print(
        "FAIL_FULL_HISTORY_STILL_DIFFERS"
    )

    print()

    print(
        "Do NOT start Paper Trading."
    )

    print(
        "Next step would be explicit numerical stabilization "
        "of slope_alignment_score followed by re-freeze."
    )


FULL_PREFIX_PARITY_DECISION = (
    "PASS_CANONICAL_FULL_HISTORY_SOLVES_PARITY"

    if FULL_PREFIX_PARITY_OK

    else

    "FAIL_FULL_HISTORY_STILL_DIFFERS"
)


## 元セルindex 73
構文状態：valid


In [ ]:
# ============================================================
# PRODUCTION LIVE ENGINE HARDENING v1
#
# Frozen Champion:
#   BASE_PLUS_REGIME
#
# PURPOSE:
#   1. Canonical full-historyをProduction必須仕様にする
#   2. tail(N)等の部分履歴による推論を禁止
#   3. Freeze時点のhistorical prefix改変をSHA256で検知
#   4. 未確定15m barでの推論を禁止
#   5. duplicate / old bar / malformed OHLCを拒否
#   6. 新しい確定15m barを安全にappendする関数を作る
#   7. Hardened Live Inference関数を作る
#   8. Runtime ContractをJSON保存
#   9. 既存Live Engineとの予測一致を確認
#
# NO TRAINING
# NO OPTIMIZATION
# NO MODEL CHANGE
# NO FEATURE CHANGE
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# ============================================================
# 0. REQUIRED STATE
# ============================================================

REQUIRED_OBJECTS = [

    # Current canonical data
    "bars",

    # Frozen Live Engine
    "LIVE_MANIFEST",
    "LIVE_FEATURES",
    "LIVE_MODEL",
    "LIVE_CALIBRATOR",
    "LIVE_CHAMPION_DIR",
    "run_live_inference",
    "normalize_live_bars",

    # Previous successful checks
    "LIVE_ENGINE_SMOKE_RESULT",
    "LIVE_ENGINE_PARITY_OK",

    # Historical replay checks
    "POLICY_PARITY_OK",
    "FULL_FEATURE_PARITY_OK",
    "PROBABILITY_PARITY_OK",
    "TRADE_TIMESTAMP_PARITY_OK",
    "TRADE_DETAIL_PARITY_OK",
    "INTEGRITY_OK",
    "PERFORMANCE_PARITY_OK",

    # Corrected sequential parity
    "FULL_PREFIX_PARITY_OK",
]


missing_objects = [

    name
    for name in REQUIRED_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Live Engine Hardeningに必要な状態が不足しています。\n\n"

        f"Missing:\n{missing_objects}\n\n"

        "Champion Freeze → Live Inference Engine → "
        "Historical Replay → Full-Prefix Parity "
        "の順に実行してください。"
    )


# ============================================================
# 1. PREVIOUS VALIDATION MUST HAVE PASSED
# ============================================================

PRIOR_VALIDATION_CHECKS = {

    "single_bar_live_parity":
        bool(
            LIVE_ENGINE_PARITY_OK
        ),

    "historical_policy":
        bool(
            POLICY_PARITY_OK
        ),

    "full_frame_features":
        bool(
            FULL_FEATURE_PARITY_OK
        ),

    "probabilities":
        bool(
            PROBABILITY_PARITY_OK
        ),

    "trade_timestamps":
        bool(
            TRADE_TIMESTAMP_PARITY_OK
        ),

    "trade_details":
        bool(
            TRADE_DETAIL_PARITY_OK
        ),

    "integrity":
        bool(
            INTEGRITY_OK
        ),

    "performance_reproduction":
        bool(
            PERFORMANCE_PARITY_OK
        ),

    # IMPORTANT:
    # The old tail(250) sequential test is superseded by this.
    "canonical_full_prefix_sequential":
        bool(
            FULL_PREFIX_PARITY_OK
        ),
}


if not all(
    PRIOR_VALIDATION_CHECKS.values()
):

    raise RuntimeError(

        "過去のProduction validationにFAILがあります。\n\n"

        f"{PRIOR_VALIDATION_CHECKS}\n\n"

        "Paper Engineへ進めません。"
    )


print("=" * 110)
print("PRIOR PRODUCTION VALIDATION")
print("=" * 110)

for key, value in PRIOR_VALIDATION_CHECKS.items():

    print(
        f"{key}: {value}"
    )


print()
print(
    "ALL PRIOR VALIDATION: PASS"
)


# ============================================================
# 2. CONSTANTS FROM FROZEN MANIFEST
# ============================================================

OHLC_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
]


def to_utc_timestamp(value):

    ts = pd.Timestamp(
        value
    )

    if ts.tzinfo is None:

        ts = ts.tz_localize(
            "UTC"
        )

    else:

        ts = ts.tz_convert(
            "UTC"
        )

    return ts


FROZEN_HISTORY_START = to_utc_timestamp(

    LIVE_MANIFEST[
        "clean_bars_start"
    ]
)


FROZEN_HISTORY_END = to_utc_timestamp(

    LIVE_MANIFEST[
        "clean_bars_end"
    ]
)


FROZEN_HISTORY_ROWS = int(

    LIVE_MANIFEST[
        "clean_bars_rows"
    ]
)


FROZEN_HISTORY_SHA256 = str(

    LIVE_MANIFEST[
        "clean_data_sha256"
    ]
)


CHAMPION_VERSION = str(

    LIVE_MANIFEST[
        "champion_version"
    ]
)


# ============================================================
# 3. CANONICAL DATA HASH
#
# Must be identical to Champion Freeze implementation.
# ============================================================

def canonical_history_sha256(
    dataframe
):

    hashed = (

        pd.util
        .hash_pandas_object(

            dataframe[
                OHLC_COLUMNS
            ],

            index=True
        )

        .values
    )


    return hashlib.sha256(

        hashed.tobytes()

    ).hexdigest()


# ============================================================
# 4. CANONICAL HISTORY VALIDATOR
#
# Production requirement:
#
#   DO NOT use tail(N)
#
#   History must begin at the original canonical clean start.
#   The frozen prefix must remain bit-for-bit identical.
# ============================================================

def validate_canonical_history(
    bars_input,
    verbose=False
):

    x = normalize_live_bars(
        bars_input
    )


    # --------------------------------------------------------
    # Must contain complete canonical origin
    # --------------------------------------------------------

    actual_start = (
        x.index.min()
    )


    actual_end = (
        x.index.max()
    )


    if (
        actual_start
        !=
        FROZEN_HISTORY_START
    ):

        raise RuntimeError(

            "CANONICAL HISTORY VIOLATION\n\n"

            "History does not begin at the frozen canonical start.\n"

            f"Expected: {FROZEN_HISTORY_START}\n"

            f"Actual:   {actual_start}\n\n"

            "Do NOT use tail(N) history for Production inference."
        )


    # --------------------------------------------------------
    # Must contain complete frozen historical prefix
    # --------------------------------------------------------

    if (
        actual_end
        <
        FROZEN_HISTORY_END
    ):

        raise RuntimeError(

            "Canonical history is incomplete.\n"

            f"Frozen prefix ends at: {FROZEN_HISTORY_END}\n"

            f"Provided history ends: {actual_end}"
        )


    frozen_prefix = (

        x.loc[
            FROZEN_HISTORY_START:
            FROZEN_HISTORY_END,
            OHLC_COLUMNS
        ]

        .copy()
    )


    # --------------------------------------------------------
    # Frozen row count must match
    # --------------------------------------------------------

    if (
        len(
            frozen_prefix
        )
        !=
        FROZEN_HISTORY_ROWS
    ):

        raise RuntimeError(

            "Frozen historical prefix row count mismatch.\n"

            f"Expected: {FROZEN_HISTORY_ROWS:,}\n"

            f"Actual:   {len(frozen_prefix):,}"
        )


    if (
        frozen_prefix.index.min()
        !=
        FROZEN_HISTORY_START
    ):

        raise RuntimeError(
            "Frozen prefix start mismatch."
        )


    if (
        frozen_prefix.index.max()
        !=
        FROZEN_HISTORY_END
    ):

        raise RuntimeError(
            "Frozen prefix end mismatch."
        )


    # --------------------------------------------------------
    # Frozen prefix hash
    #
    # Detect:
    #   edited historical prices
    #   deleted rows
    #   reordered data
    #   changed timestamps
    # --------------------------------------------------------

    prefix_hash = (

        canonical_history_sha256(
            frozen_prefix
        )
    )


    frozen_prefix_hash_ok = (

        prefix_hash
        ==
        FROZEN_HISTORY_SHA256
    )


    if not frozen_prefix_hash_ok:

        raise RuntimeError(

            "FROZEN HISTORICAL DATA HASH MISMATCH\n\n"

            "Freeze時点の過去データが変更されています。\n"

            f"Expected SHA256:\n{FROZEN_HISTORY_SHA256}\n\n"

            f"Actual SHA256:\n{prefix_hash}\n\n"

            "STOP. Production inference must not continue."
        )


    # --------------------------------------------------------
    # Forward rows
    # --------------------------------------------------------

    appended_rows = int(

        (
            x.index
            >
            FROZEN_HISTORY_END
        )
        .sum()
    )


    report = {

        "canonical_start":
            actual_start,

        "latest_bar":
            actual_end,

        "total_rows":
            int(
                len(x)
            ),

        "frozen_prefix_rows":
            int(
                len(
                    frozen_prefix
                )
            ),

        "forward_rows":
            appended_rows,

        "frozen_prefix_hash":
            prefix_hash,

        "frozen_prefix_hash_ok":
            frozen_prefix_hash_ok,

        "canonical_history_ok":
            True,
    }


    if verbose:

        print("=" * 100)
        print("CANONICAL HISTORY CHECK")
        print("=" * 100)

        for key, value in report.items():

            print(
                f"{key}: {value}"
            )


    return x, report


# ============================================================
# 5. ONE-BAR INPUT NORMALIZER
#
# Used later by API / Cloud / Paper runtime.
# ============================================================

def normalize_single_closed_bar(
    new_bar
):

    # --------------------------------------------------------
    # DataFrame
    # --------------------------------------------------------

    if isinstance(
        new_bar,
        pd.DataFrame
    ):

        if len(
            new_bar
        ) != 1:

            raise RuntimeError(
                "new_bar DataFrame must contain exactly one row."
            )

        row = new_bar.copy()


    # --------------------------------------------------------
    # Series
    # --------------------------------------------------------

    elif isinstance(
        new_bar,
        pd.Series
    ):

        if new_bar.name is None:

            raise RuntimeError(
                "Series new_bar requires timestamp as Series.name."
            )

        row = pd.DataFrame(
            [new_bar.values],
            columns=new_bar.index,
            index=[
                new_bar.name
            ]
        )


    # --------------------------------------------------------
    # Dict
    #
    # Example:
    #
    # {
    #   "timestamp": "...",
    #   "open": ...,
    #   "high": ...,
    #   "low": ...,
    #   "close": ...
    # }
    # --------------------------------------------------------

    elif isinstance(
        new_bar,
        dict
    ):

        required_keys = {

            "timestamp",
            "open",
            "high",
            "low",
            "close",
        }


        missing = (

            required_keys

            -

            set(
                new_bar.keys()
            )
        )


        if missing:

            raise RuntimeError(

                "new_bar dict missing keys: "
                f"{sorted(missing)}"
            )


        timestamp = new_bar[
            "timestamp"
        ]


        row = pd.DataFrame(

            [
                {
                    "open":
                        new_bar[
                            "open"
                        ],

                    "high":
                        new_bar[
                            "high"
                        ],

                    "low":
                        new_bar[
                            "low"
                        ],

                    "close":
                        new_bar[
                            "close"
                        ],
                }
            ],

            index=[
                timestamp
            ]
        )


    else:

        raise TypeError(

            "new_bar must be DataFrame, Series, or dict."
        )


    # --------------------------------------------------------
    # Normalize columns
    # --------------------------------------------------------

    row.columns = [

        str(c)
        .strip()
        .lower()

        for c in row.columns
    ]


    missing_ohlc = [

        c
        for c in OHLC_COLUMNS

        if c not in row.columns
    ]


    if missing_ohlc:

        raise RuntimeError(

            f"new_bar OHLC missing: {missing_ohlc}"
        )


    row = row[
        OHLC_COLUMNS
    ].copy()


    # --------------------------------------------------------
    # Timestamp
    # --------------------------------------------------------

    row.index = pd.to_datetime(

        row.index,

        utc=True,

        errors="coerce"
    )


    if row.index.isna().any():

        raise RuntimeError(
            "Invalid new_bar timestamp."
        )


    timestamp = row.index[0]


    if (

        timestamp.minute
        %
        15

        !=
        0

        or

        timestamp.second
        !=
        0

        or

        timestamp.microsecond
        !=
        0

    ):

        raise RuntimeError(

            "new_bar timestamp is not on exact 15m grid.\n"

            f"Timestamp: {timestamp}"
        )


    # --------------------------------------------------------
    # Numeric validation
    # --------------------------------------------------------

    for column in OHLC_COLUMNS:

        row[column] = pd.to_numeric(

            row[column],

            errors="coerce"
        )


    if row[
        OHLC_COLUMNS
    ].isna().any().any():

        raise RuntimeError(
            "new_bar contains NaN/non-numeric OHLC."
        )


    if (

        row[
            OHLC_COLUMNS
        ]
        <=
        0

    ).any().any():

        raise RuntimeError(
            "new_bar contains non-positive price."
        )


    open_price = float(
        row.iloc[0][
            "open"
        ]
    )

    high_price = float(
        row.iloc[0][
            "high"
        ]
    )

    low_price = float(
        row.iloc[0][
            "low"
        ]
    )

    close_price = float(
        row.iloc[0][
            "close"
        ]
    )


    if (

        high_price
        <
        max(
            open_price,
            low_price,
            close_price
        )

    ):

        raise RuntimeError(
            "Invalid HIGH in new_bar."
        )


    if (

        low_price
        >
        min(
            open_price,
            high_price,
            close_price
        )

    ):

        raise RuntimeError(
            "Invalid LOW in new_bar."
        )


    return row


# ============================================================
# 6. SAFE CANONICAL APPEND
#
# Important:
#   Existing historical data cannot be overwritten.
#
# By default:
#   gap > 15m is rejected.
#
# Later market-data layer can explicitly verify market closure
# before allowing a weekend/session gap.
# ============================================================

def append_closed_bar_to_canonical_history(

    canonical_history,

    new_bar,

    as_of_utc=None,

    allow_verified_gap=False
):

    # --------------------------------------------------------
    # Validate current state BEFORE mutation
    # --------------------------------------------------------

    history, history_report = (

        validate_canonical_history(
            canonical_history
        )
    )


    row = normalize_single_closed_bar(
        new_bar
    )


    new_time = (
        row.index[0]
    )


    last_time = (
        history.index[-1]
    )


    # --------------------------------------------------------
    # No duplicate / past modification
    # --------------------------------------------------------

    if (
        new_time
        <=
        last_time
    ):

        raise RuntimeError(

            "OLD OR DUPLICATE BAR REJECTED\n\n"

            f"Latest canonical bar: {last_time}\n"

            f"Incoming bar:         {new_time}\n\n"

            "Historical bars may not be overwritten."
        )


    # --------------------------------------------------------
    # Grid interval
    # --------------------------------------------------------

    delta_minutes = (

        new_time
        -
        last_time

    ).total_seconds() / 60.0


    if (
        delta_minutes
        <
        15.0
    ):

        raise RuntimeError(
            "Incoming bar interval < 15 minutes."
        )


    if not np.isclose(

        delta_minutes
        %
        15.0,

        0.0,

        atol=1e-12
    ):

        raise RuntimeError(

            "Incoming timestamp interval is not "
            "a multiple of 15 minutes."
        )


    gap_detected = bool(

        delta_minutes
        >
        15.0
    )


    if (

        gap_detected

        and

        not allow_verified_gap

    ):

        raise RuntimeError(

            "TIME GAP DETECTED\n\n"

            f"Previous bar: {last_time}\n"

            f"Incoming bar: {new_time}\n"

            f"Gap minutes:  {delta_minutes}\n\n"

            "Default safety policy rejects gaps.\n"

            "Backfill missing bars or explicitly verify "
            "that the gap is a legitimate market closure."
        )


    # --------------------------------------------------------
    # Closed-bar verification
    #
    # Timestamp = BAR OPEN TIME
    #
    # 10:00 bar is not usable until 10:15.
    # --------------------------------------------------------

    bar_close_time = (

        new_time

        +

        pd.Timedelta(
            minutes=15
        )
    )


    if as_of_utc is not None:

        current_time = to_utc_timestamp(
            as_of_utc
        )


        if (
            current_time
            <
            bar_close_time
        ):

            raise RuntimeError(

                "UNCLOSED BAR REJECTED\n\n"

                f"Bar open:  {new_time}\n"

                f"Bar close: {bar_close_time}\n"

                f"Now:       {current_time}"
            )


    # --------------------------------------------------------
    # Append
    # --------------------------------------------------------

    updated = pd.concat(

        [
            history,
            row
        ],

        axis=0
    )


    updated = (
        updated
        .sort_index()
    )


    # --------------------------------------------------------
    # Validate entire canonical state AFTER mutation
    # --------------------------------------------------------

    updated, updated_report = (

        validate_canonical_history(
            updated
        )
    )


    append_report = {

        "previous_latest_bar":
            last_time,

        "new_bar":
            new_time,

        "new_bar_close":
            bar_close_time,

        "delta_minutes":
            float(
                delta_minutes
            ),

        "gap_detected":
            gap_detected,

        "total_rows":
            len(
                updated
            ),

        "canonical_history_ok":
            True,
    }


    return (
        updated,
        append_report
    )


# ============================================================
# 7. HARDENED PRODUCTION INFERENCE
#
# This is the function Paper/Cloud will call.
#
# It refuses arbitrary truncated history.
# ============================================================

def run_hardened_live_inference(

    canonical_history,

    position_is_open=False,

    as_of_utc=None
):

    # --------------------------------------------------------
    # Mandatory full-history validation
    # --------------------------------------------------------

    checked_history, history_report = (

        validate_canonical_history(
            canonical_history
        )
    )


    # --------------------------------------------------------
    # Existing Frozen Live Engine
    # --------------------------------------------------------

    result = run_live_inference(

        checked_history,

        position_is_open=position_is_open,

        as_of_utc=as_of_utc
    )


    # --------------------------------------------------------
    # Add Production diagnostics
    # --------------------------------------------------------

    result[
        "canonical_history_ok"
    ] = True


    result[
        "canonical_history_rows"
    ] = int(
        len(
            checked_history
        )
    )


    result[
        "frozen_prefix_hash_ok"
    ] = bool(

        history_report[
            "frozen_prefix_hash_ok"
        ]
    )


    result[
        "forward_rows"
    ] = int(

        history_report[
            "forward_rows"
        ]
    )


    result[
        "runtime_mode"
    ] = (
        "CANONICAL_FULL_HISTORY"
    )


    return result


# ============================================================
# 8. CURRENT CANONICAL DATA VALIDATION
# ============================================================

CURRENT_CANONICAL_BARS, CURRENT_HISTORY_REPORT = (

    validate_canonical_history(

        bars,

        verbose=True
    )
)


# ============================================================
# 9. HARDENED INFERENCE SMOKE TEST
# ============================================================

print()
print("=" * 110)
print("HARDENED LIVE INFERENCE SMOKE TEST")
print("=" * 110)


HARDENED_LIVE_RESULT = (

    run_hardened_live_inference(

        CURRENT_CANONICAL_BARS,

        position_is_open=False,

        as_of_utc=None
    )
)


important_keys = [

    "signal_time",
    "signal_bar_close_time",

    "raw_p_up",
    "calibrated_p_up",
    "confidence",

    "threshold",
    "threshold_allowed",

    "session",
    "session_allowed",

    "action",
    "reason",

    "position_size",

    "canonical_history_ok",
    "canonical_history_rows",
    "frozen_prefix_hash_ok",
    "forward_rows",
    "runtime_mode",
]


for key in important_keys:

    print(
        f"{key}: "
        f"{HARDENED_LIVE_RESULT.get(key)}"
    )


# ============================================================
# 10. REGRESSION CHECK VS PREVIOUS LIVE ENGINE
#
# Hardening must NOT change prediction.
# ============================================================

old = (
    LIVE_ENGINE_SMOKE_RESULT
)

new = (
    HARDENED_LIVE_RESULT
)


HARDENING_REGRESSION_CHECKS = {

    "signal_time":

        pd.Timestamp(
            old[
                "signal_time"
            ]
        )

        ==

        pd.Timestamp(
            new[
                "signal_time"
            ]
        ),


    "raw_p_up":

        np.isclose(

            float(
                old[
                    "raw_p_up"
                ]
            ),

            float(
                new[
                    "raw_p_up"
                ]
            ),

            atol=1e-12,
            rtol=0.0
        ),


    "calibrated_p_up":

        np.isclose(

            float(
                old[
                    "calibrated_p_up"
                ]
            ),

            float(
                new[
                    "calibrated_p_up"
                ]
            ),

            atol=1e-12,
            rtol=0.0
        ),


    "confidence":

        np.isclose(

            float(
                old[
                    "confidence"
                ]
            ),

            float(
                new[
                    "confidence"
                ]
            ),

            atol=1e-12,
            rtol=0.0
        ),


    "threshold_allowed":

        bool(
            old[
                "threshold_allowed"
            ]
        )

        ==

        bool(
            new[
                "threshold_allowed"
            ]
        ),


    "session_allowed":

        bool(
            old[
                "session_allowed"
            ]
        )

        ==

        bool(
            new[
                "session_allowed"
            ]
        ),


    "action":

        str(
            old[
                "action"
            ]
        )

        ==

        str(
            new[
                "action"
            ]
        ),


    "position_size":

        np.isclose(

            float(
                old[
                    "position_size"
                ]
            ),

            float(
                new[
                    "position_size"
                ]
            ),

            atol=1e-12,
            rtol=0.0
        ),


    "canonical_hash":

        bool(
            new[
                "frozen_prefix_hash_ok"
            ]
        ),
}


HARDENING_REGRESSION_OK = all(

    HARDENING_REGRESSION_CHECKS.values()
)


print()
print("=" * 110)
print("HARDENING REGRESSION CHECK")
print("=" * 110)


for key, value in (
    HARDENING_REGRESSION_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


print()

print(
    "REGRESSION PASSED:",
    HARDENING_REGRESSION_OK
)


if not HARDENING_REGRESSION_OK:

    raise RuntimeError(

        "Hardening changed Live Engine behavior.\n"

        "STOP before Paper Trading."
    )


# ============================================================
# 11. TEST THAT tail(N) IS ACTUALLY REJECTED
#
# We WANT this test to raise internally.
# ============================================================

TAIL_HISTORY_REJECTED = False


try:

    validate_canonical_history(

        CURRENT_CANONICAL_BARS
        .tail(
            1000
        )
    )


except RuntimeError:

    TAIL_HISTORY_REJECTED = True


print()
print("=" * 110)
print("TRUNCATED HISTORY SAFETY TEST")
print("=" * 110)

print(
    "tail(1000) rejected:",
    TAIL_HISTORY_REJECTED
)


if not TAIL_HISTORY_REJECTED:

    raise RuntimeError(

        "Production safety failed: "
        "truncated history was accepted."
    )


# ============================================================
# 12. CREATE RUNTIME CONTRACT
#
# Champion manifest itself remains untouched.
# Runtime contract is saved separately.
# ============================================================

LIVE_RUNTIME_CONTRACT = {

    "schema_version":
        1,

    "champion_version":
        CHAMPION_VERSION,

    "champion_name":
        LIVE_MANIFEST[
            "champion_name"
        ],

    "runtime_history_policy":
        "CANONICAL_FULL_HISTORY_REQUIRED",

    "arbitrary_tail_history_allowed":
        False,

    "canonical_start_utc":
        FROZEN_HISTORY_START.isoformat(),

    "frozen_prefix_end_utc":
        FROZEN_HISTORY_END.isoformat(),

    "frozen_prefix_rows":
        FROZEN_HISTORY_ROWS,

    "frozen_prefix_sha256":
        FROZEN_HISTORY_SHA256,

    "frozen_prefix_must_remain_unchanged":
        True,

    "bar_interval_minutes":
        15,

    "bar_timestamp_semantics":
        "BAR_OPEN_TIME_UTC",

    "bar_must_be_closed_before_inference":
        True,

    "duplicate_bar_allowed":
        False,

    "historical_bar_overwrite_allowed":
        False,

    "time_gap_default_policy":
        "REJECT_UNLESS_EXPLICITLY_VERIFIED",

    "feature_count":
        len(
            LIVE_FEATURES
        ),

    "calibration":
        LIVE_MANIFEST[
            "calibration"
        ],

    "threshold":
        LIVE_MANIFEST[
            "threshold"
        ],

    "session":
        LIVE_MANIFEST[
            "session"
        ],

    "sizing_policy":
        LIVE_MANIFEST[
            "sizing_policy"
        ],

    "sizing_scale":
        LIVE_MANIFEST[
            "sizing_scale"
        ],

    "overlapping_positions":
        False,

    "entry_policy":
        "NEXT_15M_BAR_OPEN",

    "holding_period_minutes":
        30,

    "exit_policy":
        "FIXED_30_MINUTES",

    "cost_per_trade_return":
        LIVE_MANIFEST[
            "cost_per_trade_return"
        ],

    "validation": {

        "single_bar_parity":
            True,

        "full_frame_feature_parity":
            True,

        "canonical_prefix_sequential_parity":
            True,

        "canonical_prefix_rows_checked":
            418,

        "canonical_prefix_max_feature_diff":
            0.0,

        "probability_parity":
            True,

        "trade_timestamp_parity":
            True,

        "trade_detail_parity":
            True,

        "historical_performance_reproduction":
            True,
    },
}


# ============================================================
# 13. SAVE RUNTIME CONTRACT
# ============================================================

LIVE_CHAMPION_PATH = Path(
    LIVE_CHAMPION_DIR
)


RUNTIME_DIR = (

    LIVE_CHAMPION_PATH
    /
    "runtime"
)


RUNTIME_DIR.mkdir(

    parents=True,

    exist_ok=True
)


RUNTIME_CONTRACT_PATH = (

    RUNTIME_DIR
    /
    "live_runtime_contract.json"
)


with open(

    RUNTIME_CONTRACT_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        LIVE_RUNTIME_CONTRACT,

        f,

        ensure_ascii=False,

        indent=2
    )


# ============================================================
# 14. SAVE NOTEBOOK RUNTIME VARIABLES
# ============================================================

PRODUCTION_CANONICAL_HISTORY = (
    CURRENT_CANONICAL_BARS.copy()
)


PRODUCTION_CANONICAL_HISTORY_REPORT = (
    CURRENT_HISTORY_REPORT.copy()
)


PRODUCTION_RUNTIME_CONTRACT = (
    LIVE_RUNTIME_CONTRACT.copy()
)


PRODUCTION_RUNTIME_CONTRACT_PATH = str(
    RUNTIME_CONTRACT_PATH
)


PRODUCTION_HARDENED_LIVE_RESULT = (
    HARDENED_LIVE_RESULT.copy()
)


PRODUCTION_HARDENING_REGRESSION_OK = (
    HARDENING_REGRESSION_OK
)


PRODUCTION_TAIL_HISTORY_REJECTED = (
    TAIL_HISTORY_REJECTED
)


# ============================================================
# 15. FINAL PAPER-READINESS DECISION
# ============================================================

LIVE_HARDENING_FINAL_CHECKS = {

    "all_prior_validation":
        all(
            PRIOR_VALIDATION_CHECKS.values()
        ),

    "canonical_history_valid":
        bool(
            CURRENT_HISTORY_REPORT[
                "canonical_history_ok"
            ]
        ),

    "frozen_prefix_hash":
        bool(
            CURRENT_HISTORY_REPORT[
                "frozen_prefix_hash_ok"
            ]
        ),

    "full_prefix_parity":
        bool(
            FULL_PREFIX_PARITY_OK
        ),

    "live_regression":
        bool(
            HARDENING_REGRESSION_OK
        ),

    "truncated_history_rejected":
        bool(
            TAIL_HISTORY_REJECTED
        ),

    "runtime_contract_saved":
        bool(
            RUNTIME_CONTRACT_PATH.exists()
        ),
}


LIVE_HARDENING_OK = all(

    LIVE_HARDENING_FINAL_CHECKS.values()
)


if LIVE_HARDENING_OK:

    LIVE_HARDENING_DECISION = (

        "PASS_READY_FOR_PAPER_EXECUTION_ENGINE"
    )

else:

    LIVE_HARDENING_DECISION = (

        "FAIL_DO_NOT_BUILD_PAPER_EXECUTION_ENGINE"
    )


# ============================================================
# 16. FINAL OUTPUT
# ============================================================

print()
print("=" * 110)
print("LIVE ENGINE HARDENING FINAL REPORT")
print("=" * 110)


print(
    "Champion:",
    LIVE_MANIFEST[
        "champion_name"
    ]
)

print(
    "Version:",
    CHAMPION_VERSION
)

print(
    "Canonical rows:",
    f"{len(PRODUCTION_CANONICAL_HISTORY):,}"
)

print(
    "Canonical start:",
    PRODUCTION_CANONICAL_HISTORY.index.min()
)

print(
    "Canonical latest:",
    PRODUCTION_CANONICAL_HISTORY.index.max()
)


print()
print(
    "Runtime contract:"
)

print(
    RUNTIME_CONTRACT_PATH
)


print()
print("=" * 110)
print("FINAL CHECKS")
print("=" * 110)


for key, value in (
    LIVE_HARDENING_FINAL_CHECKS.items()
):

    print(
        f"{key}: {value}"
    )


print()

print(
    "LIVE ENGINE HARDENING PASSED:",
    LIVE_HARDENING_OK
)


print()

print(
    "FINAL DECISION:",
    LIVE_HARDENING_DECISION
)


if LIVE_HARDENING_OK:

    print()

    print(
        "Frozen Champion is now protected by "
        "a Production runtime contract."
    )

    print(
        "Truncated-history inference is prohibited."
    )

    print(
        "Frozen historical data integrity is verified by SHA256."
    )

    print(
        "Canonical full-history inference is mandatory."
    )

    print()

    print(
        "NEXT STEP:"
    )

    print(
        "BUILD PAPER EXECUTION ENGINE."
    )

else:

    print()

    print(
        "STOP."
    )

    print(
        "Do not continue to Paper Execution."
    )
